# 10 · Knowledge Distillation — Main Experiment

Full KD experiment with selected teacher and student, across 3 seeds.
Compares two student initialization strategies:

- **KD-FT**: student initialized from pretrained baseline, then KD training
- **KD-Scratch**: student initialized randomly, then KD training

**Hyperparameters:** T=4.0, alpha=0.7 (justified by ablation in notebook 09).

**Expected outcome:** KD-FT outperforms KD-Scratch because the student already has
useful pretrained features before distillation begins.


In [ ]:
# ── Mount Drive & load utils ────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import sys, shutil, os
UTILS_SRC = "/content/drive/My Drive/thesis/utils"
if os.path.exists(UTILS_SRC):
    shutil.copytree(UTILS_SRC, "/content/utils", dirs_exist_ok=True)
    sys.path.insert(0, "/content")
    print("✅ utils loaded from Drive")
else:
    sys.path.insert(0, "/content")
    print("⚠️  Place the utils/ folder at: My Drive/thesis/utils/")


In [ ]:
# ── Standard imports ────────────────────────────────────────────────
import os, time, random
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn

from utils.dataset import prepare_dataset, get_loaders, get_test_loader
from utils.models  import (
    VGG_Scratch, VGG_Pretrained,
    ResNet_Scratch, ResNet18_Pretrained,
    MobileNetV2_Scratch, MobileNetV3_Pretrained,
    count_params, model_size_mb, MODEL_REGISTRY,
)
from utils.train import (
    setup_device, set_seed, evaluate,
    train_model, train_model_three_phase,
    train_multi_seed, train_kd, plot_history,
)

device = setup_device(seed=41)


In [ ]:
# ── Dataset setup ───────────────────────────────────────────────────
prepare_dataset()


In [ ]:
SAVE_DIR = "/content/drive/My Drive/Colab Notebooks"


In [ ]:
# ── Set after running notebooks 04, 07, and 09 ───────────────────────
TEACHER_CKPT     = f"{SAVE_DIR}/resnet18_pretrained_seed_XX.pth"
STUDENT_CKPT     = f"{SAVE_DIR}/mobilenetv3_baseline_seed_XX.pth"
TEACHER_MODEL_FN = ResNet18_Pretrained
STUDENT_MODEL_FN = MobileNetV3_Pretrained

KD_TEMPERATURE = 4.0
KD_ALPHA       = 0.7
KD_EPOCHS      = 30
KD_LR          = 1e-3
KD_SEEDS       = [41, 52, 63]


In [ ]:
train_loader, val_loader = get_loaders(batch_size=64, augmentation="minimal")

teacher = TEACHER_MODEL_FN().to(device)
teacher.load_state_dict(torch.load(TEACHER_CKPT, map_location=device))
teacher_acc = evaluate(teacher, val_loader, device)
print(f"Teacher val accuracy: {teacher_acc*100:.2f}%")


In [ ]:
# ── KD-FT: initialize student from pretrained baseline ───────────────
print("\n" + "="*55 + "\nKD-FT (fine-tune from pretrained baseline)\n" + "="*55)
kd_ft_results = []

for seed in KD_SEEDS:
    print(f"\nSeed {seed}")
    set_seed(seed)
    student = STUDENT_MODEL_FN().to(device)
    student.load_state_dict(torch.load(STUDENT_CKPT, map_location=device))
    baseline_acc = evaluate(student, val_loader, device)

    best_acc, elapsed = train_kd(
        student=student, teacher=teacher,
        train_loader=train_loader, val_loader=val_loader, device=device,
        epochs=KD_EPOCHS, lr=KD_LR, weight_decay=1e-4,
        temperature=KD_TEMPERATURE, alpha=KD_ALPHA,
        patience=10, save_path=f"{SAVE_DIR}/kd_ft_seed_{seed}.pth",
    )
    kd_ft_results.append({
        "seed": seed, "baseline": baseline_acc,
        "kd_acc": best_acc, "delta": best_acc - baseline_acc, "elapsed": elapsed
    })


In [ ]:
# ── KD-Scratch: initialize student from random weights ───────────────
print("\n" + "="*55 + "\nKD-Scratch (random initialization)\n" + "="*55)
kd_scratch_results = []

for seed in KD_SEEDS:
    print(f"\nSeed {seed}")
    set_seed(seed)
    student = STUDENT_MODEL_FN().to(device)   # fresh random init — do NOT load baseline
    scratch_acc = evaluate(student, val_loader, device)
    print(f"  Random init accuracy: {scratch_acc*100:.2f}%")

    best_acc, elapsed = train_kd(
        student=student, teacher=teacher,
        train_loader=train_loader, val_loader=val_loader, device=device,
        epochs=KD_EPOCHS, lr=KD_LR, weight_decay=1e-4,
        temperature=KD_TEMPERATURE, alpha=KD_ALPHA,
        patience=10, save_path=f"{SAVE_DIR}/kd_scratch_seed_{seed}.pth",
    )
    kd_scratch_results.append({
        "seed": seed, "baseline": scratch_acc,
        "kd_acc": best_acc, "delta": best_acc - scratch_acc, "elapsed": elapsed
    })


In [ ]:
# ── Summary ──────────────────────────────────────────────────────────
import pandas as pd

def summarize(results, label):
    accs = [r["kd_acc"] for r in results]
    return {
        "Condition": label,
        "Mean Acc (%)": round(np.mean(accs)*100, 2),
        "Std (%)":      round(np.std(accs)*100,  2),
        "Best Acc (%)": round(max(accs)*100,      2),
        "Δ vs baseline (%)": round((np.mean(accs) - np.mean([r["baseline"] for r in results]))*100, 2),
    }

student_base_acc = evaluate(
    STUDENT_MODEL_FN().to(device).__class__(
        **{}
    ), val_loader, device
) if False else None  # placeholder — filled from notebook 07 results

rows = [
    {"Condition": "Student baseline (no KD)",
     "Mean Acc (%)": "→ see nb 07", "Std (%)": "—", "Best Acc (%)": "—", "Δ vs baseline (%)": "—"},
    summarize(kd_ft_results,     "KD-FT"),
    summarize(kd_scratch_results,"KD-Scratch"),
    {"Condition": "Teacher (ResNet18)",
     "Mean Acc (%)": round(teacher_acc*100, 2), "Std (%)": "—", "Best Acc (%)": "—", "Δ vs baseline (%)": "—"},
]

df = pd.DataFrame(rows).set_index("Condition")
print("\n" + df.to_string())
print("\nBest KD-FT checkpoint:     ", f"{SAVE_DIR}/kd_ft_seed_{max(kd_ft_results, key=lambda r:r['kd_acc'])['seed']}.pth")
print("Best KD-Scratch checkpoint:", f"{SAVE_DIR}/kd_scratch_seed_{max(kd_scratch_results, key=lambda r:r['kd_acc'])['seed']}.pth")
